# VeriSQL — reproduce the proof in 2 minutes

**Claim:** AI-generated SQL fails *silently* — queries that run fine and return confidently wrong answers — and a deterministic oracle can catch **and fix** those failures with zero LLM tokens.

**This notebook proves it on this machine, right now.** Run every cell (`Runtime → Run all`).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sneha21032004/veriSQL/blob/main/notebooks/verisql_proof.ipynb)

In [ ]:
# Install VeriSQL straight from GitHub (~30s)
%pip install -q "verisql[duckdb] @ git+https://github.com/Sneha21032004/veriSQL.git"
import verisql; print("VeriSQL", verisql.__version__, "ready")

## 1. A realistic warehouse — with a real-world landmine

One `NULL` in a blocklist table. That's all it takes. (Production blocklists, exclusion lists, and lookup tables accumulate NULLs constantly.)

In [ ]:
import duckdb
conn = duckdb.connect(":memory:")
conn.execute("""
    CREATE TABLE customers (id INT, name VARCHAR, region VARCHAR);
    CREATE TABLE orders (id INT, customer_id INT, amount DECIMAL(10,2),
                         status VARCHAR, created_at TIMESTAMP);
    CREATE TABLE blocklist (banned_region VARCHAR);
    INSERT INTO customers VALUES (1,'Asha','IN'),(2,'Bob','US'),(3,'Carol','UK'),(4,'Dan','CA');
    INSERT INTO orders VALUES
      (1,1,1200,'paid','2026-05-03 14:22:01'),
      (2,2,800,'paid','2026-05-15 09:10:45'),
      (3,3,300,'refunded','2026-05-20 18:05:00'),
      (4,1,950,'paid','2026-06-02 11:00:30'),
      (5,4,400,'paid','2026-04-28 08:45:12');
    INSERT INTO blocklist VALUES ('US'), (NULL);   -- <-- the landmine
""")
print("Warehouse ready. Expected answer to 'customers NOT in banned regions': Asha, Carol, Dan")

## 2. The SQL an AI naturally writes — and its silent lie

This is the textbook query. GPT/Claude/Gemini all write this shape. It parses. It runs. **It returns zero rows, always** — SQL three-valued logic: `region NOT IN (..., NULL)` is never true.

In [ ]:
naive_sql = "SELECT name FROM customers WHERE region NOT IN (SELECT banned_region FROM blocklist)"
rows = conn.execute(naive_sql).fetchall()
print(f"Naive AI SQL returned {len(rows)} rows  <-- WRONG (truth: 3 customers)")
print("A dashboard would now show: 'no unbanned customers'. Silently. Forever.")

## 3. The oracle: verify → auto-repair → re-verify

No LLM. No human. Pure AST analysis + rewrite, ~2ms.

In [ ]:
from verisql import verify_and_repair
from verisql.connectors.duckdb_conn import DuckDBConnector

db = DuckDBConnector(conn)
result = verify_and_repair(naive_sql,
                           question="Which customers are NOT in a banned region?",
                           connector=db)
print(result.summary())
print("\nCorrected SQL:\n ", result.final_sql)

In [ ]:
fixed_rows = conn.execute(result.final_sql).fetchall()
print("Repaired query returns:", [r[0] for r in fixed_rows], " <-- CORRECT")
assert [r[0] for r in fixed_rows] == ['Asha', 'Carol', 'Dan']
assert result.verified
print("\nPROOF COMPLETE: wrong answer caught and fixed autonomously, $0, no LLM.")

## 4. Two more silent failures, same treatment

In [ ]:
from verisql import verify

# (a) 'May revenue' with no date filter -> sums ALL months
r = verify("SELECT SUM(amount) FROM orders WHERE status='paid'",
           question="Total paid revenue in May 2026", connector=db)
print("(a)", r.summary().splitlines()[-1])

# (b) timestamp = date matches only exact midnight -> auto-repaired to a range
result_b = verify_and_repair("SELECT * FROM orders WHERE created_at = '2026-05-03'", connector=db)
print("(b) repaired ->", result_b.final_sql)
print("    rows before fix:", len(conn.execute("SELECT * FROM orders WHERE created_at = '2026-05-03'").fetchall()),
      "| rows after fix:", len(conn.execute(result_b.final_sql).fetchall()))

## 5. The measured benchmark

Not cherry-picked: a labeled corpus of wrong + correct queries runs live in CI.
Current numbers: **83% recall on silent failures, 91% precision.**

- Repo: https://github.com/Sneha21032004/veriSQL
- Works with DuckDB, PostgreSQL, Snowflake, BigQuery
- Also ships: tamper-evident audit trail, migration equivalence diff, MCP server so AI agents call this oracle inside their own loop

*You just reproduced the whole claim on your own machine. That's the point: deterministic means it works the same everywhere.*